# Работа с числами и строками на примере алгоритма Луна
---
М.А. Гейне (mike.geine@gmail.com)

## 1. Углубленная работа с Unicode и кодировками

Строки в Python 3 — это последовательности символов Unicode, что позволяет работать с текстами на любых языках. Однако, когда мы сохраняем строки в файлы или передаем по сети, их нужно преобразовать в байты. Этот процесс называется **кодированием**.

In [ ]:
s = "Привет, мир!"
print(f"Тип: {type(s)}, Длина: {len(s)}")

# Кодируем строку в байты, используя кодировку UTF-8
b = s.encode('utf-8')
print(f"Тип: {type(b)}, Длина: {len(b)}")
print(f"Байтовое представление: {b}")

# Декодируем байты обратно в строку
s_decoded = b.decode('utf-8')
print(f"Раскодированная строка: {s_decoded}")

# Попытка декодировать с неверной кодировкой вызовет ошибку
try:
    b.decode('ascii')
except UnicodeDecodeError as e:
    print(f"\nОшибка: {e}")

### Обработка ошибок кодирования

При работе с реальными данными часто встречаются проблемы с кодировками. Параметр `errors` помогает их обработать.

In [ ]:
# Эта байтовая строка содержит и ASCII, и не-ASCII символы (кириллица в UTF-8)
b_mixed = b'hello \xd0\xbf\xd1\x80\x69\x76\x65\x74 world'

# errors='strict' (по умолчанию) - вызовет ошибку при декодировании в ascii
# errors='replace' - заменит некорректные для ascii символы на 'U+FFFD'
print(f"'replace': {b_mixed.decode('ascii', errors='replace')}")

# errors='ignore' - просто проигнорирует (выбросит) некорректные символы
print(f"'ignore': {b_mixed.decode('ascii', errors='ignore')}")

### Нормализация и анализ символов с `unicodedata`

Этот модуль полезен для задач очистки текста, например, для удаления диакритических знаков (акцентов).

In [ ]:
import unicodedata

text = "Crème brûlée"

def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return u"".join([c for c in nfkd_form if not unicodedata.combining(c)])

cleaned_text = remove_accents(text)
print(f"Оригинал: {text}")
print(f"Без диакритики: {cleaned_text}")

# Также можно получить информацию о символе
print(f"Имя символа 'é': {unicodedata.name('é')}")

## 2. Эффективность строковых операций

Строки в Python неизменяемы (immutable). Это означает, что любая "модификация" строки на самом деле создает новую строку в памяти. Это может сильно влиять на производительность, особенно при работе с большим количеством операций.

### Конкатенация: `+` против `join()`

Давайте сравним два способа сборки строки из списка частей с помощью "магической" команды `%%timeit`, которая позволяет измерить среднее время выполнения ячейки.

In [ ]:
words = ["word"] * 10000

def concat_plus():
    result = ""
    for word in words:
        result += word
    return result

print("Измеряем время для конкатенации через `+` в цикле:")
%timeit concat_plus

In [ ]:
def concat_join():
    return "".join(words)

print("\nИзмеряем время для `join()`:")
%timeit concat_join

**Вывод:** `join()` на порядки быстрее. Он сначала вычисляет необходимый размер итоговой строки и выделяет память только один раз, в то время как `+` в цикле создает новую строку и копирует в нее данные на *каждой* итерации.

## 3. Регулярные выражения

Регулярные выражения — это мощный инструмент для поиска и манипулирования текстом на основе шаблонов.

Расшифровка:

- `U` - поиск символа U (или любого другого символа, указанного буквально)
- `a-z` - любой символ в интервале
- `[abc]` - любой из символов a, b или c
- `[^abc]` - любой из символов, кроме abc
- `.` - любой символ
- `\d` - любая цифра (0-9)
- `\D` - любой символ, кроме цифры
- `\w` - любой символ слова ([a-zA-Z0-9_])
- `\s` - любой символ пробела
- `{4}` - количество повторений предыдущего токена (4 раза)
- `+` - предыдущий токен должен повториться 1 или более раз
- `*` - предыдущий токен должен повториться 0 или более раз
- `()` - группа захвата

См. также: [https://regex101.com/](https://regex101.com/)


### Именованные группы для извлечения данных

Использование `(?P<name>...)` делает код более читаемым и надежным, так как вы обращаетесь к результатам по имени, а не по индексу.

In [ ]:
import re
log_line = '2023-10-27 10:30:00 - ERROR - User [johndoe] attempted login from IP 192.168.1.100'

# Используем именованные группы для каждого извлекаемого элемента
pattern = r'(?P<date>\d{4}-\d{2}-\d{2})\s+(?P<time>\d{2}:\d{2}:\d{2})\s+-\s+(?P<level>\w+)\s+-\s+User\s+\[(?P<username>\w+)\].*IP\s+(?P<ip>\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})'

match = re.search(pattern, log_line)
if match:
    log_data = match.groupdict()
    print(log_data)
    print(f"Пользователь {log_data['username']} совершил действие с уровнем {log_data['level']}")

### "Болтливые" (verbose) регулярные выражения

Для сложных шаблонов можно использовать флаг `re.VERBOSE` (или `re.X`), чтобы добавлять пробелы и комментарии, делая выражение понятным.

In [ ]:
email_pattern_verbose = re.compile(r"""
    ^                           # Начало строки
    [a-zA-Z0-9._%+-]+           # Имя пользователя
    @                           # Символ @
    [a-zA-Z0-9.-]+              # Имя домена
    \.                         # Точка (экранированная)
    [a-zA-Z]{2,}                # Домен верхнего уровня (2+ символа)
    $                           # Конец строки
""", re.VERBOSE)

email = "test.user+alias@example.com"
if email_pattern_verbose.match(email):
    print("Корректный email")

### Просмотр вперед/назад (Lookarounds)

Lookarounds позволяют проверять наличие символов до или после совпадения, *не включая* их в само совпадение. Это полезно для извлечения данных в определенном контексте.

- `(?=...)` - Positive Lookahead (просмотр вперед): ищет совпадение, только если *за ним* следует `...`.
- `(?<=...)` - Positive Lookbehind (просмотр назад): ищет совпадение, только если *перед ним* есть `...`.

In [ ]:
text = "Товар А стоит $10.99, а Товар Б - $15.50. Скидка 5%."

# Извлечь только числа, которые являются ценами (т.е. после знака $)
# (?<=\$) - это просмотр назад, который проверяет, что перед числом есть '$'
prices = re.findall(r'(?<=\$)\d+\.\d+', text)
print(f"Найденные цены: {prices}")

# Найти числа, за которыми следует знак процента
# (?=%) - это просмотр вперед, который проверяет, что после числа идет '% '
discounts = re.findall(r'\d+(?=%)', text)
print(f"Найденные скидки: {discounts}")

## Алгоритм Луна

Алгоритм Луна (Luhn algorithm) — это алгоритм контрольной суммы, используемый для проверки различных идентификационных номеров, таких как номера кредитных карт, номера социального страхования (SIN) в Канаде и некоторых других странах.

Как он работает:

1. Удвоение цифр: Начиная с конца номера (справа налево), удвоить каждую вторую цифру.
2. Суммирование цифр: Если удвоение цифры приводит к двузначному числу, сложить цифры этого числа (например, 12 -> 1 + 2 = 3).
3. Суммирование всех цифр: Сложить все цифры, полученные на предыдущих шагах, включая неудвоенные.
4. Проверка результата: Если полученная сумма делится на 10 без остатка, то номер считается действительным по алгоритму Луна.

In [ ]:
card_number = '4111-1111-4555-1142'

0. Подготовка номера карты

In [ ]:
card_translation = str.maketrans({'-': '', ' ': ''})
translated_card_number = card_number.translate(card_translation)
translated_card_number

1. Сумма нечётных чисел

In [ ]:
card_number_reversed = translated_card_number[::-1]
card_number_reversed

In [ ]:
odd_digits = card_number_reversed[::2]
odd_digits

In [ ]:
sum_of_odd_digits = 0

for digit in odd_digits:
  sum_of_odd_digits += int(digit)
sum_of_odd_digits

2. Сумма чётных чисел

In [ ]:
even_digits = card_number_reversed[1::2]
even_digits

In [ ]:
sum_of_even_digits = 0

for digit in even_digits:
  number = int(digit) * 2
  if number >= 10:
    number = (number // 10) + (number % 10)
  sum_of_even_digits += number

sum_of_even_digits



3. Подсчёт суммы

In [ ]:
total = sum_of_even_digits + sum_of_odd_digits
total

In [ ]:
total % 10 == 0

Оформим в функции

In [ ]:
def clear_card_number(card_number):
  card_translation = str.maketrans({'-': '', ' ': ''})
  return card_number.translate(card_translation)

def verify_luhn(str_of_digits):
  sum_of_odd_digits = 0
  card_number_reversed = str_of_digits[::-1]
  odd_digits = card_number_reversed[::2]

  for digit in odd_digits:
      sum_of_odd_digits += int(digit)

  sum_of_even_digits = 0
  even_digits = card_number_reversed[1::2]
  for digit in even_digits:
      number = int(digit) * 2
      if number >= 10:
          number = (number // 10) + (number % 10)
      sum_of_even_digits += number
  total = sum_of_odd_digits + sum_of_even_digits
  return total % 10 == 0

In [ ]:
card_number = '4111-1111-4555-1142'
clean = clear_card_number(card_number)
verify_luhn(clean)


In [ ]:
card_number = '4111-1111-4555-1143'
clean = clear_card_number(card_number)
verify_luhn(clean)

### Можно ли написать лучше?

Вариант решения в функциональном стиле:

In [ ]:
from functools import reduce

def compress(digit):
  return (digit // 10) + (digit % 10) if digit > 9 else digit

def verify_luhn2(str_of_digits):
  ints = list(map(lambda c: int(c), clean[::-1]))
  sum = reduce(lambda acc, pair: acc + (compress(pair[1]*2) if pair[0] % 2 == 0 else pair[1]), enumerate(ints, 1), 0)
  return sum % 10 == 0

In [ ]:
card_number = '4111-1111-4555-1142'
clean = clear_card_number(card_number)
verify_luhn2(clean)

In [ ]:
card_number = '4111-1111-4555-1143'
clean = clear_card_number(card_number)
verify_luhn2(clean)